# SQLi Sentinel — CodeBERT fine-tuning (Colab GPU)

Runtime → Change runtime type → **T4 GPU**. Upload the repo as `SQLxW.zip` (without `.venv`, `node_modules`) or clone it from your git remote.

In [ ]:
# Option A: upload a zip of the repo
from google.colab import files
uploaded = files.upload()  # choose SQLxW.zip
!unzip -q -o SQLxW.zip -d /content
%cd /content/SQLxW

In [ ]:
# Option B: clone from git
# !git clone https://github.com/<you>/SQLxW.git /content/SQLxW
# %cd /content/SQLxW

In [ ]:
!pip install -q -e 'backend[ml]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build the dataset (synthetic + optional reviewed real-world samples)

In [ ]:
!python ml/data/build_dataset.py
# with reviewed real samples:
# !python ml/data/build_dataset.py --real ml/data/real/candidates.jsonl

## 2. Fine-tune
Around 20–30 minutes on a T4 at `max_length=512`.

In [ ]:
!python ml/train.py --max-length 512 --batch-size 16 --epochs 4 --fp16 \
    --augment-prob 0.5 --label-smoothing 0.1 --dropout 0.1 --freeze-layers 6 --patience 2

## 3. Evaluate all detectors (regex vs AST vs CodeBERT vs ensemble)

In [ ]:
!python ml/evaluate.py --splits test heldout --probe-renaming
!cat ml/reports/eval.md

## 4. Download the trained model
Unzip into `artifacts/model/` locally and restart the backend.

In [ ]:
!cd artifacts && zip -qr /content/sqli-sentinel-model.zip model
files.download('/content/sqli-sentinel-model.zip')